# Sequential Process

CrewAI learning notebook — companion to the roadmap. Platform: [Agent Foundry](https://agent-foundry-theta.vercel.app)


In [ ]:
!pip install -q crewai crewai-tools


In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


## Three-agent sequential pipeline

We use **researcher → analyst → writer**. With `Process.sequential`, task 1 completes first; its output is passed as context to task 2, then task 3 builds on task 2. The **`tasks` list order is the execution order**.


In [ ]:
from crewai import Agent, Task, Crew, Process

researcher = Agent(
    role="Researcher",
    goal="Collect accurate facts on the topic.",
    backstory="You gather concise, factual bullets.",
    verbose=True,
)

analyst = Agent(
    role="Analyst",
    goal="Derive insights from the research.",
    backstory="You turn raw notes into clear takeaways.",
    verbose=True,
)

writer = Agent(
    role="Writer",
    goal="Produce a short executive summary.",
    backstory="You write tight, readable prose.",
    verbose=True,
)

research_task = Task(
    description="Research: main benefits and risks of {topic}.",
    expected_output="Bullet list of benefits and risks.",
    agent=researcher,
)

analyze_task = Task(
    description="Using the research above, state the top 3 takeaways and one open question.",
    expected_output="Three one-sentence takeaways and one open question.",
    agent=analyst,
)

write_task = Task(
    description="Write a 2–3 paragraph executive summary from the analyst output.",
    expected_output="Executive summary only.",
    agent=writer,
)

crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analyze_task, write_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff(inputs={"topic": "CrewAI sequential crews"})
print(result)


## Key takeaways

- **Sequential** means tasks run **in the order they appear** in the `tasks` list.
- **Later tasks receive prior task outputs** as context automatically.
- Use **`process=Process.sequential`** (also the **default** in typical setups).
- Choose sequential for **fixed pipelines**; use **hierarchical** when a **manager** should **delegate** dynamically.
